In [1]:
import time, json, warnings
import numpy as np, pandas as pd
from pathlib import Path
warnings.filterwarnings("ignore")
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    average_precision_score, roc_auc_score, matthews_corrcoef, confusion_matrix)

P = Path("processed"); B = P / "balanced"
X_val = np.load(P / "X_val.npy"); y_val = np.load(P / "y_val.npy")

sets = ["none", "class_weight", "random_over", "random_under", "smote",
        "borderline_smote", "adasyn", "under_then_smote",
        "smote_tomek", "smote_enn", "tomek_links"]

In [2]:
rows, cms = [], {}
for s in sets:
    d = np.load(B / ("none.npz" if s == "class_weight" else f"{s}.npz"))
    X, y = d["X"], d["y"]
    m = DecisionTreeClassifier(criterion="gini", splitter="best", max_depth=None,
                               random_state=42,
                               class_weight="balanced" if s == "class_weight" else None)
    t = time.time(); m.fit(X, y); fit_s = time.time() - t

    pred = m.predict(X_val); prob = m.predict_proba(X_val)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_val, pred).ravel()
    cms[s] = [[int(tn), int(fp)], [int(fn), int(tp)]]
    rows.append({
        "set": s, "train_rows": len(y),
        "accuracy": accuracy_score(y_val, pred),
        "precision": precision_score(y_val, pred),
        "recall_attack": recall_score(y_val, pred),
        "normal_recall": tn / (tn + fp),
        "f1": f1_score(y_val, pred),
        "macro_f1": f1_score(y_val, pred, average="macro"),
        "pr_auc": average_precision_score(y_val, prob),
        "roc_auc": roc_auc_score(y_val, prob),
        "mcc": matthews_corrcoef(y_val, pred),
        "FPR": fp / (fp + tn), "FNR": fn / (fn + tp),
        "depth": int(m.get_depth()), "leaves": int(m.get_n_leaves()), "fit_s": fit_s})
    print(s, "done")

results = pd.DataFrame(rows).sort_values("mcc", ascending=False)
results.round(4)

none done
class_weight done
random_over done
random_under done
smote done
borderline_smote done
adasyn done
under_then_smote done
smote_tomek done
smote_enn done
tomek_links done


,set,train_rows,accuracy,precision,recall_attack,normal_recall,f1,macro_f1,pr_auc,roc_auc,mcc,FPR,FNR,depth,leaves,fit_s
10,tomek_links,140193,0.9925,0.9954,0.9959,0.9708,0.9957,0.9840,0.9950,0.9839,0.9680,0.0292,0.0041,33,1297,2.2756
0,none,140311,0.9923,0.9953,0.9957,0.9703,0.9955,0.9835,0.9950,0.9838,0.9669,0.0297,0.0043,33,1313,2.2467
8,smote_tomek,242092,0.9905,0.9957,0.9933,0.9725,0.9945,0.9798,0.9950,0.9836,0.9597,0.0275,0.0067,37,2306,5.2034
4,smote,242568,0.9901,0.9956,0.9929,0.9723,0.9943,0.9791,0.9950,0.9835,0.9582,0.0277,0.0071,42,2592,4.9582
9,smote_enn,236452,0.9899,0.9954,0.9929,0.9706,0.9941,0.9785,0.9944,0.9817,0.9571,0.0294,0.0071,27,811,4.6450
1,class_weight,140311,0.9894,0.9956,0.9922,0.9718,0.9939,0.9776,0.9952,0.9846,0.9554,0.0282,0.0078,34,1315,2.1230
2,random_over,242568,0.9893,0.9956,0.9920,0.9720,0.9938,0.9774,0.9952,0.9844,0.9548,0.0280,0.0080,30,1309,4.9140
5,borderline_smote,242568,0.9889,0.9954,0.9917,0.9711,0.9936,0.9766,0.9945,0.9820,0.9533,0.0289,0.0083,49,3700,7.5906
6,adasyn,242847,0.9868,0.9955,0.9892,0.9713,0.9923,0.9722,0.9942,0.9809,0.9447,0.0287,0.0108,57,4334,7.2040
7,under_then_smote,76108,0.9796,0.9959,0.9805,0.9742,0.9881,0.9583,0.9937,0.9788,0.9180,0.0258,0.0195,43,1381,1.2373


In [3]:
results.round(4).to_csv(P / "dt_results.csv", index=False)
json.dump(cms, open(P / "dt_confusion.json", "w"))

In [4]:
lr = pd.read_csv(P / "lr_results.csv").set_index("set")
sv = pd.read_csv(P / "svm_results.csv").set_index("set")
xg = pd.read_csv(P / "xgb_results.csv").set_index("set")
dt = results.set_index("set")
cmp = pd.DataFrame({"LR_mcc": lr["mcc"], "SVM_mcc": sv["mcc"],
                    "XGB_mcc": xg["mcc"], "DT_mcc": dt["mcc"],
                    "LR_FNR": lr["FNR"], "SVM_FNR": sv["FNR"],
                    "XGB_FNR": xg["FNR"], "DT_FNR": dt["FNR"]}).round(4)
cmp.sort_values("XGB_mcc", ascending=False)

,LR_mcc,SVM_mcc,XGB_mcc,DT_mcc,LR_FNR,SVM_FNR,XGB_FNR,DT_FNR
set,,,,,,,,
none,0.8351,0.8269,0.9782,0.9669,0.0124,0.0128,0.0002,0.0043
tomek_links,0.8351,0.8269,0.9780,0.9680,0.0127,0.0128,0.0004,0.0041
smote,0.7381,0.7386,0.9762,0.9582,0.0779,0.0756,0.0017,0.0071
smote_tomek,0.7387,0.7402,0.9759,0.9597,0.0776,0.0748,0.0019,0.0067
smote_enn,0.7469,0.7460,0.9711,0.9571,0.0728,0.0711,0.0029,0.0071
class_weight,0.7372,0.7376,0.9671,0.9554,0.0784,0.0768,0.0045,0.0078
under_then_smote,0.7320,0.7358,0.9671,0.9180,0.0810,0.0780,0.0044,0.0195
random_over,0.7387,0.7382,0.9664,0.9548,0.0777,0.0754,0.0045,0.0080
random_under,0.7309,0.7340,0.9541,0.9069,0.0813,0.0788,0.0087,0.0240
